In [12]:
# type: ignore
import cv2
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


# ==========================================================
# CELL 1 : LOAD IMAGE
# ==========================================================

folder_path = r"C:\Users\proha\OneDrive\Desktop\computer vision\RAF30JAN2026047471009900060SSANSTUC00GTDB"

band3 = cv2.imread(folder_path + r"\BAND3.tif", cv2.IMREAD_UNCHANGED)

print("Original shape:", band3.shape)

band3 = cv2.normalize(
    band3,
    None,
    0,
    255,
    cv2.NORM_MINMAX
).astype(np.uint8)

# resize for speed
band3_small = cv2.resize(
    band3,
    None,
    fx=0.5,
    fy=0.5
)

print("Working shape:", band3_small.shape)

Original shape: (16330, 18274)
Working shape: (8165, 9137)


In [13]:
# ==========================================================
# BEST SATELLITE-FIELD SUPPRESSION VERSION
# Removes farmland rectangles / repetitive field grids
# Keeps real straight engineered lines
# Use this as CELL 2 (Heatmap creation)
# ==========================================================

# Improve image contrast first
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
enhanced_full = clahe.apply(band3_small)

# Stronger edges
edges = cv2.Canny(enhanced_full, 50, 150)

# Detect lines
lines = cv2.HoughLinesP(
    edges,
    1,
    np.pi/180,
    threshold=120,
    minLineLength=120,
    maxLineGap=5
)

# Empty heatmap
heatmap = np.zeros_like(band3_small, dtype=np.float32)

H, W = band3_small.shape

if lines is not None:

    for line in lines:

        x1, y1, x2, y2 = line[0]

        dx = x2 - x1
        dy = y2 - y1

        length = np.sqrt(dx*dx + dy*dy)

        if length < 80:
            continue

        angle = abs(np.degrees(np.arctan2(dy, dx)))

        # --------------------------------------------------
        # Reject farmland/grid lines:
        # short horizontal / vertical repeated edges
        # --------------------------------------------------
        if (angle < 12 or angle > 168 or (78 < angle < 102)) and length < 180:
            continue

        # --------------------------------------------------
        # Reject border artifacts
        # --------------------------------------------------
        margin = 12

        if (x1 < margin and x2 < margin) or \
           (y1 < margin and y2 < margin) or \
           (x1 > W-margin and x2 > W-margin) or \
           (y1 > H-margin and y2 > H-margin):
            continue

        # --------------------------------------------------
        # Draw valid line
        # --------------------------------------------------
        cv2.line(
            heatmap,
            (x1, y1),
            (x2, y2),
            1.0,
            5
        )

# Smooth
heatmap = cv2.GaussianBlur(heatmap, (3,3), 0)

# Remove weak zones
heatmap[heatmap < 0.20] = 0

# Normalize
if heatmap.max() > 0:
    heatmap = heatmap / heatmap.max()

print("Improved field-suppressed heatmap created.")

Improved field-suppressed heatmap created.


In [14]:
# ==========================================================
# CELL X : THRESHOLD + DILATION (YOUR STEP 2 + 3)
# ==========================================================

# Convert HEATMAP to 0–255
heatmap_uint8 = (heatmap * 255).astype(np.uint8)

# Threshold (>200)
_, binary_mask = cv2.threshold(
    heatmap_uint8,
    80,
    255,
    cv2.THRESH_BINARY
)

# Dilate (circle radius ≈10)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31))
dilated_mask = cv2.dilate(binary_mask, kernel)

print("Threshold + dilation done.")

Threshold + dilation done.


In [15]:
# ==========================================================
# STAGGERING FUNCTION
# Applies stagger ONLY in 16x16 regions
# where overlap with mask >= 80%
# ==========================================================

def apply_staggering(patch, mask, patch_id):

    staggered = patch.copy().astype(np.float32)

    H, W = patch.shape

    # ------------------------------------------------------
    # Divide patch into 16x16 sub-blocks
    # ------------------------------------------------------
    block_size = 16

    count_blocks = 0

    for bi in range(0, H, block_size):
        for bj in range(0, W, block_size):

            # ----------------------------------------------
            # Extract local block
            # ----------------------------------------------
            patch_block = patch[
                bi:bi+block_size,
                bj:bj+block_size
            ]

            mask_block = mask[
                bi:bi+block_size,
                bj:bj+block_size
            ]

            # Skip incomplete edge blocks
            if patch_block.shape != (16,16):
                continue

            # ----------------------------------------------
            # Compute overlap INSIDE local block
            # ----------------------------------------------
            overlap = np.sum(mask_block > 0) / mask_block.size

            print(f"Block ({bi},{bj}) overlap = {overlap:.2f}")

            # ----------------------------------------------
            # Apply staggering ONLY if overlap >= 80%
            # ----------------------------------------------
            if overlap >= 0.8:

                count_blocks += 1

                print(
                      f"Patch {patch_id} ->" 
                      f"STAGGER APPLIED at block ({bi},{bj})"
                )

                for i in range(1, block_size-1):
                    for j in range(1, block_size-1):

                        global_i = bi + i
                        global_j = bj + j

                        # Modify ONLY masked pixels
                        if mask[global_i, global_j] > 0:

                            # Direction-aware staggering
                            gx = (
                                int(patch[global_i, global_j+1]) -
                                int(patch[global_i, global_j-1])
                            )

                            gy = (
                                int(patch[global_i+1, global_j]) -
                                int(patch[global_i-1, global_j])
                            )

                            # Horizontal structure
                            if abs(gx) > abs(gy):

                                neighbors = [
                                    patch[global_i, global_j-1],
                                    patch[global_i, global_j+1]
                                ]

                            # Vertical structure
                            else:

                                neighbors = [
                                    patch[global_i-1, global_j],
                                    patch[global_i+1, global_j]
                                ]

                            neighbors = [n for n in neighbors if n > 0]

                            if len(neighbors) > 0:

                                # Stronger visible stagger
                                staggered[
                                    global_i,
                                    global_j
                                ] = np.mean(neighbors) * 0.5

    print("Blocks with overlap >= 0.8 :", count_blocks)                            

    return staggered.astype(np.uint8)

In [5]:
# ==========================================================
# STAGGERING FUNCTION (DIRECTION-AWARE — FINAL)
# ==========================================================

#def apply_staggering(patch, mask):

    #staggered = patch.copy().astype(np.float32)

   # H, W = patch.shape

    #for i in range(1, H-1):
    #    for j in range(1, W-1):

    #        if mask[i, j] > 0:

    #            # Detect direction using gradient
    #            gx = patch[i, j+1] - patch[i, j-1]
    #            gy = patch[i+1, j] - patch[i-1, j]

    #            # Choose direction
    #            if abs(gx) > abs(gy):
    #                neighbors = [patch[i, j-1], patch[i, j+1]]  # horizontal
    #            else:
    #                neighbors = [patch[i-1, j], patch[i+1, j]]  # vertical

    #            neighbors = [n for n in neighbors if n > 0]

    #            if len(neighbors) > 0:
    #                staggered[i, j] = np.mean(neighbors)

   # return staggered.astype(np.uint8)

In [16]:
# ==========================================================
# PATCHES WITH MASK
# ==========================================================

def create_patches_with_mask(
    image,
    heatmap,
    mask,
    patch_size=128,
    stride=128
):

    patches = []

    count_patches = 0

    h, w = image.shape

    for i in range(0, h - patch_size + 1, stride):
        for j in range(0, w - patch_size + 1, stride):

            # ------------------------------------------------
            # Extract patches
            # ------------------------------------------------
            img_patch = image[
                i:i+patch_size,
                j:j+patch_size
            ]

            heat_patch = heatmap[
                i:i+patch_size,
                j:j+patch_size
            ]

            mask_patch = mask[
                i:i+patch_size,
                j:j+patch_size
            ]

            # ------------------------------------------------
            # PATCH OVERLAP CHECK
            # ------------------------------------------------
            overlap = (
                np.sum(mask_patch > 0)
                / mask_patch.size
            )

            print(
                f"Patch index = {len(patches)} | overlap = {overlap}"
            )

            # ------------------------------------------------
            # Apply staggering if patch contains
            # detected stagger region
            # ------------------------------------------------
            if overlap > 0:

                count_patches += 1

                img_patch = apply_staggering(
                    img_patch,
                    mask_patch,
                    len(patches)
                )

            # ------------------------------------------------
            # Store patch
            # ------------------------------------------------
            patches.append(
                (
                    (i, j),
                    img_patch,
                    heat_patch,
                    mask_patch
                )
            )

    print(
    "Patches crossing threshold :",
    count_patches
    )

    return patches


# ==========================================================
# RUN PATCH CREATION
# ==========================================================

patches = create_patches_with_mask(
    band3_small,
    heatmap,
    dilated_mask
)

print("Patches ready:", len(patches))

Patch index = 0 | overlap = 0.0
Patch index = 1 | overlap = 0.0
Patch index = 2 | overlap = 0.0
Patch index = 3 | overlap = 0.0
Patch index = 4 | overlap = 0.0
Patch index = 5 | overlap = 0.0
Patch index = 6 | overlap = 0.0
Patch index = 7 | overlap = 0.0
Patch index = 8 | overlap = 0.0
Patch index = 9 | overlap = 0.0
Patch index = 10 | overlap = 0.0
Patch index = 11 | overlap = 0.0
Patch index = 12 | overlap = 0.0
Patch index = 13 | overlap = 0.0
Patch index = 14 | overlap = 0.0
Patch index = 15 | overlap = 0.30682373046875
Block (0,0) overlap = 0.00
Block (0,16) overlap = 0.00
Block (0,32) overlap = 0.00
Block (0,48) overlap = 0.00
Block (0,64) overlap = 0.00
Block (0,80) overlap = 0.64
Block (0,96) overlap = 1.00
Patch 15 ->STAGGER APPLIED at block (0,96)
Block (0,112) overlap = 0.78
Block (16,0) overlap = 0.00
Block (16,16) overlap = 0.00
Block (16,32) overlap = 0.00
Block (16,48) overlap = 0.00
Block (16,64) overlap = 0.00
Block (16,80) overlap = 0.89
Patch 15 ->STAGGER APPLIED at

In [ ]:
# ==========================================================
# CELL 3 : CREATE PATCHES AFTER HEATMAP
# ==========================================================

#def create_patches(image, heatmap, patch_size=128, stride=128):

#    patches = []

#    h, w = image.shape

#    for i in range(0, h - patch_size + 1, stride):
 #       for j in range(0, w - patch_size + 1, stride):

 #           img_patch = image[i:i+patch_size, j:j+patch_size]
 #           heat_patch = heatmap[i:i+patch_size, j:j+patch_size]

 #           patches.append(((i, j), img_patch, heat_patch))

 #   return patches


#patches = create_patches(
 #   band3_small,
 #   heatmap,
 #   patch_size=128,
 #   stride=128
#)

#print("Total patches:", len(patches))

In [8]:
# ==========================================================
# STEP 4 + 5 : SUB-PATCH CHECK (16x16)
# ==========================================================

def should_stagger(mask_patch):

    sub_size = 16
    H, W = mask_patch.shape

    for i in range(0, H, sub_size):
        for j in range(0, W, sub_size):

            sub = mask_patch[i:i+sub_size, j:j+sub_size]

            if sub.size == 0:
                continue

            overlap = np.sum(sub > 0) / sub.size

            if overlap >= 0.8:
                return True

    return False

In [ ]:
# ==========================================================
# STEP 6 : APPLY STAGGERING
# ==========================================================

#def apply_staggering(patch):

  #  staggered = patch.copy().astype(np.float32)

   # H, W = patch.shape

    #for i in range(1, H-1):
     #   for j in range(1, W-1):

     #       if patch[i, j] == 0:  # void pixel (you can refine condition)

      #          neighbors = patch[i-1:i+2, j-1:j+2].flatten()
      #          neighbors = neighbors[neighbors > 0]

      #          if len(neighbors) > 0:
      #              staggered[i, j] = np.mean(neighbors)

   # return staggered.astype(np.uint8)

In [9]:
# ==========================================================
# CELL 4 : INTERACTIVE VIEWER
# ==========================================================

def show_patch(idx):

    # =========================================
    # GET PATCH
    # =========================================
    (i, j), staggered_patch, heat_patch, mask_patch = patches[idx]

    # Original image patch
    raw_patch = band3_small[i:i+128, j:j+128]

    # =========================================
    # DISPLAY
    # =========================================
    plt.figure(figsize=(18,6))

    # -----------------------------------------
    # ORIGINAL IMAGE
    # -----------------------------------------
    plt.subplot(1,3,1)

    low = np.percentile(raw_patch, 2)
    high = np.percentile(raw_patch, 98)

    plt.imshow(
     raw_patch,
     cmap='gray',
     vmin=low,
     vmax=high
)

    plt.title(f"Original Patch\nLocation ({i},{j})")
    plt.axis("off")

    # -----------------------------------------
    # DETECTED STAGGER ZONES
    # -----------------------------------------
    plt.subplot(1,3,2)

    low = np.percentile(raw_patch, 2)
    high = np.percentile(raw_patch, 98)

    plt.imshow(
     raw_patch,
     cmap='gray',
     vmin=low,
     vmax=high
)

    low = np.percentile(raw_patch, 2)
    high = np.percentile(raw_patch, 98)

    plt.imshow(
     raw_patch,
     cmap='gray',
     vmin=low,
     vmax=high
)

    plt.title("Detected Stagger Zones")
    plt.axis("off")

    # -----------------------------------------
    # FINAL STAGGERED IMAGE
    # -----------------------------------------
    plt.subplot(1,3,3)

    low = np.percentile(raw_patch, 2)
    high = np.percentile(raw_patch, 98)

    plt.imshow(
     raw_patch,
     cmap='gray',
     vmin=low,
     vmax=high
)

    plt.title("After Applying Stagger")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [10]:
# ==========================================================
# CELL 5 : SLIDER VIEWER
# ==========================================================

widgets.interact(
    show_patch,
    idx=widgets.IntSlider(
        min=0,
        max=len(patches)-1,
        step=1,
        value=0,
        description='Patch:',
        continuous_update=False
    )
)

NameError: name 'patches' is not defined